# Supervised Machine Learning: Linear Regression

## Linear Regression
In this demo, we follow the ML process:
1. **Remember:** Load and inspect the data.
2. **Formulate:** Build a linear regression model first on raw (unscaled) data.
3. **Predict:** Evaluate the model's performance.

Then we apply feature scaling and rebuild the model to compare results.
We use the Student Performance dataset from Kaggle to predict the "Performance Index" of students.

In [29]:
# Import neccesary libraries
import pandas as pd
import numpy as np

# Download data from Kaggle
import kagglehub

# Download latest version
path = kagglehub.dataset_download("nikhil7280/student-performance-multiple-linear-regression")
print("Path to dataset files:", path)

# Import dataframe
df = pd.read_csv(f"{path}/Student_Performance.csv") #datafile from Kaggle website where it is named
df

Path to dataset files: /Users/juliadunn/.cache/kagglehub/datasets/nikhil7280/student-performance-multiple-linear-regression/versions/1


,Hours Studied,Previous Scores,Extracurricular Activities,Sleep Hours,Sample Question Papers Practiced,Performance Index
0,7,99,Yes,9,1,91.0
1,4,82,No,4,2,65.0
2,8,51,Yes,7,2,45.0
3,5,52,Yes,5,2,36.0
4,7,75,No,8,5,66.0
...,...,...,...,...,...,...
9995,1,49,Yes,4,2,23.0
9996,7,64,Yes,8,5,58.0
9997,6,83,Yes,8,5,74.0
9998,9,97,Yes,7,0,95.0


In [30]:
df.info() # no "null" values in the dataset

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 6 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   Hours Studied                     10000 non-null  int64  
 1   Previous Scores                   10000 non-null  int64  
 2   Extracurricular Activities        10000 non-null  object 
 3   Sleep Hours                       10000 non-null  int64  
 4   Sample Question Papers Practiced  10000 non-null  int64  
 5   Performance Index                 10000 non-null  float64
dtypes: float64(1), int64(4), object(1)
memory usage: 468.9+ KB


In [31]:
df.describe() # gives a statistical summary of the dataset
# cannot currently do linear regression for extracurricular activities, so we need to convert string data and encode it as a number

,Hours Studied,Previous Scores,Sleep Hours,Sample Question Papers Practiced,Performance Index
count,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000
mean,4.992900,69.445700,6.530600,4.583300,55.224800
std,2.589309,17.343152,1.695863,2.867348,19.212558
min,1.000000,40.000000,4.000000,0.000000,10.000000
25%,3.000000,54.000000,5.000000,2.000000,40.000000
50%,5.000000,69.000000,7.000000,5.000000,55.000000
75%,7.000000,85.000000,8.000000,7.000000,71.000000
max,9.000000,99.000000,9.000000,9.000000,100.000000


In [32]:
df['Extracurricular Activities'].value_counts()

Extracurricular Activities
No     5052
Yes    4948
Name: count, dtype: int64

In [ ]:
# Convert extracurricular activities to numeric 
df['Extracurricular Activities'] = df['Extracurricular Activities'].map({'Yes': 1,
                                      'No': 0})

# Define the features and target variable based on the dataset
feature_vars = ['Hours Studied', 'Previous Scores', 'Extracurricular Activities', 
                'Sleep Hours', 'Sample Question Papers Practiced']
X = pd.DataFrame(df[feature_vars])
y = pd.Series(df['Performance Index'])
# Target: Performance Index
print(y.head())
# Display a preview of the dataset
print(X.head())

0    91.0
1    65.0
2    45.0
3    36.0
4    66.0
Name: Performance Index, dtype: float64
   Hours Studied  Previous Scores  Extracurricular Activities  Sleep Hours  \
0              7               99                           1            9   
1              4               82                           0            4   
2              8               51                           1            7   
3              5               52                           1            5   
4              7               75                           0            8   

   Sample Question Papers Practiced  
0                                 1  
1                                 2  
2                                 2  
3                                 2  
4                                 5  


## Part 1: Linear Regression on Unscaled Data
In this section, we build a [linear regression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html#sklearn.linear_model.LinearRegression.fit) model on the raw data.
This helps us see the effect of differing scales on the coefficients (next week).
We start by [spliting our data into training and testing sets](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html#sklearn.model_selection.train_test_split).

In [42]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

# Split the raw data (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42) # indices for X dataframe and y series are aligned

# Initialize and train the linear regression model on unscaled data
lin_reg = LinearRegression() # need to have an object to borrow the class, allows you to run many over and over again (lin_reg1 lin_reg2 etc.)
lin_reg.fit(X_train, y_train)

# Make predictions on the test set
y_pred = lin_reg.predict(X_test)
y_pred # should have 2000 inputs

array([54.71185392, 22.61551294, 47.90314471, ..., 16.79341955,
       63.34327368, 45.94262301])

In [ ]:
from sklearn.metrics import mean_squared_error, root_mean_squared_error, r2_score
# Evaluate model performance
mse = mean_squared_error(y_test, y_pred)
rmse = root_mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("Unscaled Data Model:")
print(f"Mean Squared Error: {mse:.2f}") 
print(f"Root Squared Error: {rmse:.2f}") # essentially +- error
print(f"R² Score: {r2:.2f}") # close to 1, good job 

Unscaled Data Model:
Mean Squared Error: 4.08
Root Squared Error: 2.02
R² Score: 0.99


### Notes on Unscaled Model:
- **Coefficients (Unscaled):**
    - Each coefficient represents the change in the Performance Index for a one-unit change in the respective feature, holding all other features constant.
    - For example, if "Hours Studied" has a coefficient of 2.85, it implies that for each additional hour studied, the Performance Index increases by 2.85 points (assuming other factors remain constant).
    - However, because features are in different units (e.g., hours vs. scores), comparing these coefficients directly may be misleading.

- **R² Score:**
    - This metric indicates the proportion of the variance in the target variable explained by the model.
    - An R² close to 1 suggests a very good fit, while an R² near 0 indicates the model fails to capture much variance.

- **MSE & RMSE:**
    - MSE measures the average squared difference between actual and predicted values.
    - RMSE, being the square root of MSE, gives an error metric in the same units as the target.
    - Lower RMSE values indicate better predictive performance.

In [48]:
# View our model's coefficients (parameters)
print("Model Coefficients (Unscaled):")
print(pd.Series(lin_reg.coef_,
                index = X.columns)) 
# y = B0 + B1X1 + B2X2 + B3X3 + B4X4 + B5X5

print("\nModel Intercept (Unscaled):")
print(lin_reg.intercept_)


Model Coefficients (Unscaled):
Hours Studied                       2.852484
Previous Scores                     1.016988
Extracurricular Activities          0.608617
Sleep Hours                         0.476941
Sample Question Papers Practiced    0.191831
dtype: float64

Model Intercept (Unscaled):
-33.92194621555482


### Manually Computing a Prediction from Our Model
- In this section, we'll calculate a predicted value by hand (i.e., by multiplying the model's coefficients by the original feature values and adding the intercept).
- This mirrors exactly what the model does internally.

- **Why is this helpful?**
   - It reinforces how linear regression makes its predictions using the equation: `prediction = intercept + (coef_1 * x_1) + (coef_2 * x_2) + ...`
   - It helps us see the individual impact of each feature on the final prediction.
   - It confirms that the manual approach matches the `model.predict()` output.

#### 1. Extract the coefficients and intercept from our trained model

In [ ]:
# Extract coefs and intercept


print("Coefficients (Unscaled):")
print("\nIntercept:")

#### 2. Select a single row of our data (e.g., the second row)
- We select only the columns that were used as features in our model.
- The row's values represent the actual data for Hours Studied, Previous Scores, etc.

In [ ]:
# This row's feature values will be multiplied by our coefficients.
# for demonstration
# features only
print("Feature values (Row", , "):\n", )

#### 3. Compute the manual prediction

In [ ]:
# Manually predicting value
print("\nManual Prediction for Row", , ":", )

**Explanation:**
- We multiply each feature value by its corresponding coefficient and sum them up.
- Then, we add the intercept.
- This is precisely the linear regression equation:
$$
\hat{y} = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \dots + \beta_n x_n
$$

Where:
 - $\beta_0$ is the intercept
 - $\beta_i$ is the coefficient for feature $x_i$

 Thus, `manual_prediction` should match what the model would predict internally.

#### 4. Compare to `model.predict()` for confirmation

In [ ]:
# Extract model prediction
print("Model Prediction from lin_reg.predict():", )

### **Observation:**
- The `manual_prediction` and `model_prediction` should be nearly identical (up to minor floating-point differences).
- If they match, we've confirmed our understanding of how the model uses coefficients and intercept to make a prediction.

### Why This Matters
- **Transparency:** It shows exactly how each feature influences the final predicted value.
- **Verification:** Confirms our "manual" math aligns with the model's internal computation.
- **Interpretability:** By inspecting the coefficients, we see which features have the biggest impact (positive or negative) on the Performance Index, and we can discuss whether the magnitudes make sense given the domain context.

# Conclusion
In this demo, we:
- Built and evaluated a linear regression model on unscaled data.
- Re-trained the model after applying feature scaling.
- Observed that while overall performance metrics (**MSE** and **R²**) may be similar, scaling is crucial for the interpretability of model coefficients and for ensuring that features contribute in a balanced way.
  
### Key Takeaways:
- **Coefficients:** On unscaled data, coefficients are tied to the original units, which can be hard to compare.
  After scaling, coefficients represent the effect of a one standard deviation change in the feature.
- **R² Score:** Reflects the proportion of variance in the target variable explained by the model.
- **MSE (and RMSE):** Lower values indicate better model performance; RMSE provides an error measure in the target's units.

This process reflects the "remember-formulate-predict" approach in machine learning.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

def ols_summary(X, y, lin_model=None, scaled=False, test_size=0.2, random_state=42):

    # Train/test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )

    # Scale if requested
    if scaled:
        scaler = StandardScaler()
        X_train = pd.DataFrame(scaler.fit_transform(X_train),
                               columns=X.columns, index=X_train.index)
        X_test  = pd.DataFrame(scaler.transform(X_test),
                               columns=X.columns, index=X_test.index)

    n_train, p = X_train.shape
    n_test = X_test.shape[0]

    # Fit on train only
    model = lin_model.fit(X_train, y_train)

    # --- Training stats (for inference) ---
    y_pred_train = model.predict(X_train)
    res_train = y_train - y_pred_train
    ss_res_train = np.sum(res_train**2)
    ss_tot_train = np.sum((y_train - np.mean(y_train))**2)

    # --- Test stats (honest performance) ---
    y_pred_test = model.predict(X_test)
    res_test = y_test - y_pred_test
    ss_res_test = np.sum(res_test**2)
    ss_tot_test = np.sum((y_test - np.mean(y_test))**2)

    r2_train = 1 - ss_res_train / ss_tot_train
    r2_test  = 1 - ss_res_test  / ss_tot_test
    r2_adj   = 1 - (1 - r2_test) * (n_test - 1) / (n_test - p - 1)

    # F-statistic (based on test)
    ss_reg = ss_tot_test - ss_res_test
    f_stat = (ss_reg / p) / (ss_res_test / (n_test - p - 1))
    f_pval = 1 - stats.f.cdf(f_stat, p, n_test - p - 1)

    # Log-likelihood / AIC / BIC (based on test)
    log_lik = -n_test/2 * np.log(2 * np.pi * ss_res_test / n_test) - n_test/2
    aic = 2 * (p + 1) - 2 * log_lik
    bic = np.log(n_test) * (p + 1) - 2 * log_lik

    # Coefficient inference (based on train — this is where model was fit)
    MSE = ss_res_train / (n_train - p - 1)
    X_b = np.column_stack([np.ones(n_train), X_train])
    se = np.sqrt(np.diag(MSE * np.linalg.inv(X_b.T @ X_b)))
    coefs = np.concatenate([[model.intercept_], model.coef_])
    t_stats_vals = coefs / se
    p_values = 2 * (1 - stats.t.cdf(np.abs(t_stats_vals), df=n_train - p - 1))
    ci_low  = coefs - 1.96 * se
    ci_high = coefs + 1.96 * se

    names = ["const"] + X.columns.tolist()

    # Print
    print("=" * 78)
    print(f"{'OLS Regression Results':^78}")
    print("=" * 78)
    print(f"{'R-squared (test):':<45}{r2_test:.3f}")
    print(f"{'R-squared (train):':<45}{r2_train:.3f}")
    print(f"{'Adj. R-squared (test):':<45}{r2_adj:.3f}")
    print(f"{'F-statistic:':<45}{f_stat:.3f}")
    print(f"{'Prob (F-statistic):':<45}{f_pval:.5f}")
    print(f"{'Log-Likelihood:':<45}{log_lik:.3f}")
    print(f"{'No. Observations (train/test):':<45}{n_train}/{n_test}")
    print(f"{'AIC:':<45}{aic:.2f}")
    print(f"{'BIC:':<45}{bic:.2f}")
    print(f"{'Df Residuals:':<45}{n_test - p - 1}")
    print(f"{'Df Model:':<45}{p}")
    print("=" * 78)
    print(f"{'':16}{'coef':>10}{'std err':>10}{'t':>10}{'P>|t|':>10}{'[0.025':>10}{'0.975]':>10}")
    print("-" * 78)
    for name, c, s, t, pv, lo, hi in zip(names, coefs, se, t_stats_vals, p_values, ci_low, ci_high):
        print(f"{name:<16}{c:>10.4f}{s:>10.3f}{t:>10.3f}{pv:>10.3f}{lo:>10.3f}{hi:>10.3f}")
    print("=" * 78)

In [ ]:
ols_summary(X, y, lin_model = LinearRegression(), scaled = True)

                            OLS Regression Results                            
R-squared (test):                            0.989
R-squared (train):                           0.989
Adj. R-squared (test):                       0.989
F-statistic:                                 35800.758
Prob (F-statistic):                          0.00000
Log-Likelihood:                              -4244.618
No. Observations (train/test):               8000/2000
AIC:                                         8501.24
BIC:                                         8534.84
Df Residuals:                                1994
Df Model:                                    5
                      coef   std err         t     P>|t|    [0.025    0.975]
------------------------------------------------------------------------------
const              55.2408     0.023  2418.470     0.000    55.196    55.286
Hours Studied       7.3856     0.023   324.007     0.000     7.341     7.430
Previous Scores    17.6369     0.023 

In [ ]:
ols_summary(X, y, lin_model = LinearRegression(), scaled = False)

                            OLS Regression Results                            
R-squared (test):                            0.989
R-squared (train):                           0.989
Adj. R-squared (test):                       0.989
F-statistic:                                 35800.758
Prob (F-statistic):                          0.00000
Log-Likelihood:                              -4244.618
No. Observations (train/test):               8000/2000
AIC:                                         8501.24
BIC:                                         8534.84
Df Residuals:                                1994
Df Model:                                    5
                      coef   std err         t     P>|t|    [0.025    0.975]
------------------------------------------------------------------------------
const             -33.9219     0.143  -237.380     0.000   -34.202   -33.642
Hours Studied       2.8525     0.009   324.007     0.000     2.835     2.870
Previous Scores     1.0170     0.001 